# ViT vs CNN on HAM10000 — Colab runner

Run the cells from top to bottom. Before you start: **Runtime → Change runtime type → GPU**.

- **First session:** downloads HAM10000 from Kaggle, builds `metadata.csv`, the two
  splits and the 256 px image cache, and saves them to Google Drive (`MyDrive/ham10000_data`).
- **Later sessions:** the data cell just copies that cache back (seconds), so the
  splits are always exactly the same.
- **Training outputs** go to the `out_root` folder set in each config
  (`MyDrive/ham10000_runs/<run_name>/`). If Colab disconnects, run the training cell
  again: it resumes from `last.pt`.

## 1. Settings

In [ ]:
import os
import shutil
import subprocess

REPO_URL = "https://github.com/himaallu/ham10000-vit-vs-cnn.git"
BRANCH = "claude/loving-thompson-8feyjy"   # change to "main" once the branch is merged
REPO_DIR = "/content/ham10000-vit-vs-cnn"

DATA_DIR = "/content/data"                              # fast local disk, wiped every session
DRIVE_DATA = "/content/drive/MyDrive/ham10000_data"     # kept: kaggle.json, CSVs, image cache
LRCHECK_ROOT = "/content/drive/MyDrive/ham10000_lrcheck"

CONFIG = "configs/run1_seed0.yaml"   # the run to train in section 6

# Make Python scripts print each line straight away (otherwise epoch lines
# only appear when the script ends).
os.environ["PYTHONUNBUFFERED"] = "1"


def run(cmd):
    """Run a shell command, show its output live, and stop if it fails.

    (A plain `!command` cell keeps going after an error; this doesn't.)
    """
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}")

## 2. Mount Google Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime type!")

## 3. Get the code and install requirements

If the repository is private, add a GitHub token in Colab's **Secrets** panel (key icon
on the left) named `GITHUB_TOKEN`, and allow this notebook to use it.

In [ ]:
# Use a GitHub token from Colab Secrets if there is one (needed for a private repo).
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.exists(REPO_DIR):
    # Already cloned in this session: get the latest version of the branch.
    run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
    run(f"git -C {REPO_DIR} checkout -q {BRANCH}")
    run(f"git -C {REPO_DIR} pull -q origin {BRANCH}")
else:
    run(f"git clone -q --branch {BRANCH} {url} {REPO_DIR}")

os.chdir(REPO_DIR)
run("git log --oneline -1")              # which version of the code is running
run("pip install -q -r requirements.txt")

## 4. Data (the slow part runs only once)

The Kaggle key is only needed the first time. Get it from Kaggle → Settings → API →
**Create New Token**; you upload it once and it is kept on Drive.

In [ ]:
os.makedirs(DRIVE_DATA, exist_ok=True)
CACHE_TAR = f"{DRIVE_DATA}/images_256.tar"
CSV_FILES = ["metadata.csv", "split_lesion.csv", "split_image.csv"]

if not os.path.exists(CACHE_TAR):
    # Kaggle key: from Drive if saved before, otherwise upload it now.
    drive_key = f"{DRIVE_DATA}/kaggle.json"
    if not os.path.exists(drive_key):
        from google.colab import files
        print("Upload your kaggle.json")
        uploaded = files.upload()
        with open(drive_key, "wb") as f:
            f.write(uploaded["kaggle.json"])
    os.makedirs("/root/.kaggle", exist_ok=True)
    shutil.copy(drive_key, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)   # Kaggle refuses keys others can read

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(CACHE_TAR):
    # Later sessions: copy the saved CSVs and resized images back from Drive.
    print("Restoring data from Drive ...")
    for name in CSV_FILES:
        shutil.copy(f"{DRIVE_DATA}/{name}", f"{DATA_DIR}/{name}")
    run(f"tar -xf {CACHE_TAR} -C {DATA_DIR}")
else:
    # First session: download, build metadata, split, resize, then save to Drive.
    run(f"python src/data.py --data_dir {DATA_DIR}")
    run(f"python src/splits.py --data_dir {DATA_DIR}")
    run(f"python src/preprocess.py --data_dir {DATA_DIR}")
    print("Saving CSVs and image cache to Drive ...")
    run(f"tar -cf {CACHE_TAR} -C {DATA_DIR} images_256")   # one big file copies much faster than 10k small ones
    for name in CSV_FILES:
        shutil.copy(f"{DATA_DIR}/{name}", f"{DRIVE_DATA}/{name}")

# Check: one cached image per row of metadata.csv.
import pandas as pd
n_rows = len(pd.read_csv(f"{DATA_DIR}/metadata.csv"))
n_images = len(os.listdir(f"{DATA_DIR}/images_256"))
print(f"metadata rows: {n_rows}, cached images: {n_images}")
assert n_rows == n_images, "image cache does not match metadata.csv"

## 5. Quick learning-rate check (optional)

For each model: 3 learning rates × 3 epochs on the lesion split (standard mode, seed 0),
**train and val only** (`--skip_test`, so the test set plays no part in choosing
settings). With only 3 epochs the cosine schedule is squeezed, so this is a rough
comparison, not a final result. Expect roughly 30–60 min on a T4 for all 6 runs.

Finished checks are skipped when you rerun the cell. Checkpoints are deleted afterwards
to save Drive space (about 1 GB each for ViT); only `history.csv` is kept.

In [ ]:
import yaml

LRS = [3e-5, 1e-4, 3e-4]
BASE_CONFIGS = {"bit": "configs/run1_seed0.yaml", "vit": "configs/run2_seed0.yaml"}

for model, base in BASE_CONFIGS.items():
    for lr in LRS:
        cfg = yaml.safe_load(open(base))
        name = f"lrcheck_{model}_lr{lr:g}"
        cfg.update(lr=lr, epochs=3, run_name=name, out_root=LRCHECK_ROOT)
        out_dir = f"{LRCHECK_ROOT}/{name}"
        history = f"{out_dir}/history.csv"
        if os.path.exists(history) and len(pd.read_csv(history)) == cfg["epochs"]:
            print(f"{name}: already done, skipping")
            continue

        cfg_path = f"/content/{name}.yaml"
        with open(cfg_path, "w") as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
        print(f"\n===== {name} =====")
        run(f"python src/train.py --config {cfg_path} --skip_test")

        # Delete the big checkpoints once this check has finished.
        for ckpt in ["best.pt", "last.pt"]:
            if os.path.exists(f"{out_dir}/{ckpt}"):
                os.remove(f"{out_dir}/{ckpt}")

In [ ]:
# Val macro-F1 per epoch for every check (higher is better).
rows = []
for model in BASE_CONFIGS:
    for lr in LRS:
        history = f"{LRCHECK_ROOT}/lrcheck_{model}_lr{lr:g}/history.csv"
        if os.path.exists(history):
            h = pd.read_csv(history)
            for _, r in h.iterrows():
                rows.append({"model": model, "lr": lr, "epoch": int(r["epoch"]),
                             "val_macro_f1": r["val_macro_f1"], "train_loss": r["train_loss"]})
checks = pd.DataFrame(rows)
print("Val macro-F1 by epoch:")
print(checks.pivot_table(index=["model", "lr"], columns="epoch", values="val_macro_f1").round(3))
print("\nTrain loss by epoch:")
print(checks.pivot_table(index=["model", "lr"], columns="epoch", values="train_loss").round(3))

## 6. Train one run

Set `CONFIG` in section 1 (e.g. `configs/run2_seed0.yaml`) and run this cell. Outputs go to
`MyDrive/ham10000_runs/<run_name>/`. Running it again resumes an interrupted run, or just
redoes the test predictions if it already finished.

In [ ]:
run(f"python src/train.py --config {CONFIG}")

import json
cfg = yaml.safe_load(open(CONFIG))
out_dir = f"{cfg['out_root']}/{cfg['run_name']}"
print(f"\nOutputs in {out_dir}: {sorted(os.listdir(out_dir))}")
print(json.dumps(json.load(open(f"{out_dir}/summary.json")), indent=2))